In [ ]:
import scipy.io
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_regression
from sklearn.preprocessing import StandardScaler  # 添加缺失的导入
from sklearn.ensemble import RandomForestRegressor  # 添加缺失的导入
from sklearn.multioutput import MultiOutputRegressor  # 添加缺失的导入

# 设置 Times New Roman 字体
plt.rcParams['font.sans-serif'] = ['Times New Roman']  # 使用 Times New Roman 字体
plt.rcParams['axes.unicode_minus'] = False  # 解决负号 '-' 显示为方块的问题

# 1. 加载数据
print("正在加载数据...")
df_augmented = pd.read_excel('../data/CTGAN_data_multi.xlsx')
df_original = pd.read_excel('../data/new_data_x_names.xlsx')
print(f"增强数据集大小: {df_augmented.shape}")
print(f"原始数据集大小: {df_original.shape}")

# 2. 定义目标变量
target_vars = ['x25', 'x26', 'x27', 'x28', 'x29']
# 3. 分离特征和目标
# 从增强数据中分离
X_augmented = df_augmented.drop(target_vars, axis=1)
y_augmented = df_augmented[target_vars]

# 从原始数据中分离（用于最终评估）
X_original = df_original.drop(target_vars, axis=1)
y_original = df_original[target_vars]

# 4. 分割原始数据为测试集
X_train_orig, X_test_orig, y_train_orig, y_test_orig = train_test_split(
    X_original, y_original, test_size=0.2, random_state=42
)

# 5. 数据标准化
scaler_X = StandardScaler()
X_augmented_scaled = scaler_X.fit_transform(X_augmented)
X_test_scaled = scaler_X.transform(X_test_orig)  # 使用相同的缩放器

# 6. 创建并训练模型
print("\n开始训练随机森林模型...")
model = MultiOutputRegressor(RandomForestRegressor(
    n_estimators=300,
    max_depth=30,
    min_samples_split=2,
    min_samples_leaf=1,
    max_features=None,
    n_jobs=-1,
    random_state=42
))

# 使用增强数据训练
model.fit(X_augmented_scaled, y_augmented)
print("模型训练完成！")

# 7. 在测试集上进行预测
y_pred = model.predict(X_test_scaled)

# 获取真实值和预测值 (选择第一个目标变量 x26 进行可视化)
target_idx = 0  # 选择目标变量的索引 (0=x26, 1=x27, ...)
target_name = target_vars[target_idx]
print(f"可视化目标变量: {target_name}")

truths1 = y_test_orig.iloc[:, target_idx].values  # 获取真实值
predictions1 = y_pred[:, target_idx]  # 获取预测值

print(f"truth1 shape: {truths1.shape}")
print(f"predictions shape: {predictions1.shape}")



In [ ]:
# 确保 truths1 和 predictions1 是一维数组
truths1 = truths1.squeeze()
predictions1 = predictions1.squeeze()

r2 = r2_score(truths1, predictions1)
print(f"R^2 值为: {r2}")

# 只取前 500 个点
num_points = 200
if len(truths1) > num_points:
    truths1 = truths1[:num_points]
    predictions1 = predictions1[:num_points]

# 计算残差和标准差
residuals = truths1 - predictions1
sigma = np.std(residuals)

# 计算置信区间
conf_intervals = {
    'mu ± sigma': (predictions1 - sigma, predictions1 + sigma),
    'mu ± 2sigma': (predictions1 - 2 * sigma, predictions1 + 2 * sigma),
    'mu ± 3sigma': (predictions1 - 3 * sigma, predictions1 + 3 * sigma),
}

# 绘制图像
test_indices = np.arange(len(truths1))
plt.figure(figsize=(12, 5))

# 绘制真实测试数据
plt.scatter(test_indices, truths1, color='red', label=f'{target_name}', s=5, zorder=5)

# 绘制预测均值
plt.plot(test_indices, predictions1, color='blue', label='Prediction($\\mu$)', zorder=4)

# 绘制置信区间
plt.fill_between(test_indices, conf_intervals['mu ± sigma'][0], conf_intervals['mu ± sigma'][1],
                 color='navy', alpha=0.6, label='$\\mu \\pm \\sigma$')
plt.fill_between(test_indices, conf_intervals['mu ± 2sigma'][0], conf_intervals['mu ± 2sigma'][1],
                 color='deepskyblue', alpha=0.5, label='$\\mu \\pm 2\\sigma$')
plt.fill_between(test_indices, conf_intervals['mu ± 3sigma'][0], conf_intervals['mu ± 3sigma'][1],
                 color='green', alpha=0.3, label='$\\mu \\pm 3\\sigma$')

plt.xlim(left=0, right=len(truths1))

# 设置图表标题和标签
plt.xlabel('k', fontsize=20, labelpad=-10)
plt.ylabel('Output', fontsize=20)
plt.legend()
# 设置x轴和y轴的刻度大小
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)

# 缩小图例
plt.legend(prop={'size': 12})
# 保存图像为PNG文件
plt.savefig(f"TrueFalse_{target_name}.png", format='png', bbox_inches='tight', dpi=600)
plt.show()